In [1]:
import numpy as np
import pymc as pm
import arviz as az
# -- use this line at the beginning of your notebook to turn on interactive plots
%matplotlib notebook
# note these imports, thay may be useful in your own task

import matplotlib.pyplot as plt  # plotting library
import numpy as np  # work with numeric arrays without labeled axes
import xarray as xr  # work with arrays with labeled axes
import pandas as pd

In [2]:
xlsx_file = 'Final_full_dataset.xlsx'
df = pd.read_excel(xlsx_file)
#df_cleaned1 = df.rename(columns={'PlasmaCurrent_Measured_ND': 'I_p'})
df['pre_ts'] = df['pre_ts'] /100
df['PlasmaCurrent_Measured_ND'] = df['PlasmaCurrent_Measured_ND'] /(-50000)
df['W'] = df['W'] /1000
df['elongation_lcfs'] = df['elongation_lcfs'] *2
df['dw'] = (df['dw'] / 20)
df['H_alpha'] = df['H_alpha'] *(-1)
df['Average_triangularity'] = df['Average_triangularity'] *10
df['f_ELM'] = df['f_ELM'] *10
df['H_alpha'] = df['H_alpha'] *10
df['beta_n'] = df['beta_n'] *5
df['q95'] = df['q95'].abs()
df['NBI'] = (df['NBI'] /20)+0.01
df['resistance'] = (df['resistance'].abs())*100
df['ohmic_power'] =( df['ohmic_power'].abs() )/40000
df['B_phi_R_mag'] = df['B_phi_R_mag'].abs()*5
df = df.apply(pd.to_numeric, errors='coerce')
selected_columns = ['pre_ts', "pe_tem","pe_den",'PlasmaCurrent_Measured_ND', 'CorrectedDensity', 'W', 'dw', 'beta_n', 'q95', 'elongation_lcfs', 'f_ELM', 'B_phi_R_mag', 'H_alpha', 'Average_triangularity',"NBI","plasma_seconds_from_boronization","ohmic_power","resistance"]
log_df = df[selected_columns].apply(np.log)
log_df.columns = [f"log_{col}" for col in selected_columns]
name=['log_PlasmaCurrent_Measured_ND', 'log_CorrectedDensity', 'log_W', 'log_dw', 'log_beta_n', 'log_q95', 'log_elongation_lcfs', 'log_f_ELM', 'log_B_phi_R_mag', 'log_H_alpha', 'log_Average_triangularity',"log_NBI","log_plasma_seconds_from_boronization","log_ohmic_power","log_resistance"]
X = log_df[name]  # 自变量列
X['constant'] = 1
name2=['PlasmaCurrent_Measured_ND', 'CorrectedDensity', 'W', 'dw', 'beta_n', 'q95', 'elongation_lcfs', 'f_ELM', 'B_phi_R_mag', 'H_alpha', 'Average_triangularity',"NBI","plasma_seconds_from_boronization","ohmic_power","resistance"]
X2 = df[name2]  # 自变量列'log_W', 'log_PlasmaCurrent_Measured_ND', 
y = log_df['log_pre_ts']
# 因变量列
y2 = df['pre_ts']
X


/var/folders/5g/7cwc8n193kz9d4h5lc3ky_0c0000gn/T/ipykernel_16955/852024748.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['constant'] = 1


,log_PlasmaCurrent_Measured_ND,log_CorrectedDensity,log_W,log_dw,log_beta_n,log_q95,log_elongation_lcfs,log_f_ELM,log_B_phi_R_mag,log_H_alpha,log_Average_triangularity,log_NBI,log_plasma_seconds_from_boronization,log_ohmic_power,log_resistance,constant
0,1.754865,1.835160,1.994021,2.642247,1.639976,1.035687,1.288407,1.382302,1.937616,0.336416,1.346678,1.802557,2.772589,1.965485,1.502601,1
1,1.720854,1.719335,1.991094,2.718432,1.674108,1.065600,1.282343,1.313044,1.930903,0.152634,1.342393,1.894558,2.833213,1.911700,-1.144183,1
2,1.728394,1.743708,1.966934,2.601153,1.646222,1.045546,1.282211,0.409473,1.928227,0.300984,1.322644,1.951503,2.890372,1.642844,2.368607,1
3,1.720188,1.954939,2.217218,3.281348,1.905296,1.064120,1.279498,0.400478,1.918437,0.371178,1.383530,1.960320,2.890372,1.832061,1.343690,1
4,1.724230,2.019050,2.239866,3.560832,1.917983,1.070835,1.279394,0.348140,1.916670,1.586368,1.385134,1.960320,2.890372,1.929488,-0.149612,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138,1.885336,1.707537,2.026936,3.007287,1.537978,0.947417,1.301589,1.402424,1.952096,0.490166,1.334222,2.388896,3.761200,2.130614,1.643883,1
139,1.893165,1.552164,1.799577,2.372460,1.301895,0.927998,1.301935,2.111965,1.957312,1.734710,1.295824,2.407660,3.761200,2.191266,1.253120,1
140,1.502230,1.561486,1.926601,2.382077,1.774064,1.343801,1.275524,1.309333,1.983507,0.961012,1.408283,-4.605170,3.218876,1.459206,0.950907,1
141,1.721392,1.818295,2.211301,2.412077,1.837558,1.132765,1.281965,0.809681,1.990488,1.370877,1.364062,-4.605170,3.583519,1.742740,-0.207653,1


In [3]:


# Step 1: Generate simulated data
np.random.seed(0)
N = 3000  # Number of samples
D = 16  # Number of predictors
'''true_beta = np.array([2.5, 0.0, -1.5, 1.0, 5.0])  # True regression coefficients, sparse
sigma_true = 1.0  # True noise standard deviation

# Generate predictors
X = np.random.randn(N, D)

# Generate response variable y
noise = np.random.normal(0, sigma_true, N)
y = X @ true_beta + noise'''

# Step 2: Define the PyMC model
with pm.Model() as model:
    # Priors
    # Beta for regression coefficients, grouped into classes
    alpha = pm.HalfNormal("alpha", sigma=1.0)  # Dirichlet prior concentration parameter
    p = pm.Dirichlet("p", a=pm.math.ones(16) * alpha, shape=16)  # Class probabilities

    S = pm.Categorical("S", p=p, shape=D)  # Class assignment for predictors (0-9)

    eta = pm.InverseGamma("eta", alpha=1.0, beta=1.0)  # Variance for beta_m
    beta_m = pm.Normal("beta_m", mu=0.7, sigma=eta, shape=16)  # Regression coefficients for each class

    # Ensure S only indexes valid categories of beta_m
    beta = pm.Deterministic("beta", beta_m[S])  # Map beta based on S

    # Sparsity indicator gamma
    pi = pm.Beta("pi", alpha=1.0, beta=1.0)  # Probability of gamma=1
    gamma = pm.Bernoulli("gamma", p=pi, shape=D)

    # Noise variance (sigma^2)
    sigma2 = pm.InverseGamma("sigma2", alpha=0.1, beta=0.1)  # Prior for noise variance

    # Observed data likelihood
    mu = pm.math.dot(X, beta * gamma)  # Predicted mean
    y_obs = pm.Normal("y_obs", mu=mu, sigma=pm.math.sqrt(sigma2), observed=y)

    # Inference
    trace = pm.sample(N, tune=500, return_inferencedata=True)

# Step 3: Analyze and visualize the results
az.plot_trace(trace, var_names=["beta", "gamma", "sigma2", "eta", "pi", "p"])
plt.show()
summary=az.summary(trace, var_names=["beta", "gamma", "sigma2", "eta", "pi", "p"])
print(summary)

Multiprocess sampling (4 chains in 4 jobs)
CompoundStep
>NUTS: [alpha, p, eta, beta_m, pi, sigma2]
>CategoricalGibbsMetropolis: [S]
>BinaryGibbsMetropolis: [gamma]


Output()

ValueError: Not enough samples to build a trace.

In [8]:
beta_post_mean = trace.posterior['beta'].mean(dim=["chain", "draw"]).values
gamma_post_mean = trace.posterior['gamma'].mean(dim=["chain", "draw"]).values
gamma_binary = (gamma_post_mean > 0.5).astype(int)
print(gamma_binary)
sigma2_post_mean = trace.posterior['sigma2'].mean().values

y_pred_mean = X @ (beta_post_mean * gamma_post_mean)  # 预测均值
print(y)
noise_samples = np.random.normal(0, np.sqrt(sigma2_post_mean.mean()), size=y.shape[0])
y_pred = y_pred_mean + noise_samples  
print(y_pred)
# 计算 R²
SS_res = np.sum((y_pred - y) ** 2)
SS_tot = np.sum((y - np.mean(y)) ** 2)
R_squared = 1 - (SS_res / SS_tot)
print(f"模型的 R² 值为: {R_squared:.4f}")

# 绘制实际值与预测值的散点图
plt.figure(figsize=(8,6))
plt.scatter(y, y_pred, alpha=0.5)
plt.xlabel("实际值 y")
plt.ylabel("预测值 $\hat{y}$")
plt.title("实际值与预测值的关系图")
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')  # 绘制 y=x 的参考线
plt.show()

[0 1 0 1 0 0 1 1 0 0 0 0 0 0 0 0]
0      2.034706
1      2.586259
2      2.457021
3      3.141130
4      2.234306
         ...   
138    2.574094
139    1.980572
140    1.960647
141    2.109116
142    2.109116
Name: log_pre_ts, Length: 143, dtype: float64
0      4.468528
1      3.960417
2      4.322718
3      5.086136
4      5.113795
         ...   
138    4.171317
139    3.306168
140    2.970999
141    3.891934
142    3.477440
Length: 143, dtype: float64
模型的 R² 值为: -6.9504


<>:22: SyntaxWarning: invalid escape sequence '\h'
<>:22: SyntaxWarning: invalid escape sequence '\h'
/var/folders/5g/7cwc8n193kz9d4h5lc3ky_0c0000gn/T/ipykernel_10755/3720505427.py:22: SyntaxWarning: invalid escape sequence '\h'
  plt.ylabel("预测值 $\hat{y}$")


<IPython.core.display.Javascript object>

In [32]:
# 从后验中随机抽取 10 个样本
num_samples = 1000
posterior_indices = np.random.choice(
    trace.posterior["beta"].stack(samples=["chain", "draw"]).shape[1], 
    size=num_samples, 
    replace=False
)

# 定义新数据
X_new =X  # 新数据 (10 个样本)

# 存储预测结果
beta_new_look = []
predictions = []
S_new = []
gamma_new=[]
for idx in posterior_indices:
    beta_sample = trace.posterior["beta"].stack(samples=["chain", "draw"]).values[:, idx]
    gamma_sample = trace.posterior["gamma"].stack(samples=["chain", "draw"]).values[:, idx]
    S_sample = trace.posterior["S"].stack(samples=["chain", "draw"]).values[:, idx]
    sigma_sample = np.sqrt(trace.posterior["sigma2"].stack(samples=["chain", "draw"]).values[idx])
    #gamma_binary = (gamma_post_mean > 0.5).astype(int)
    # 计算预测值
    mu_pred = X @ (beta_sample * gamma_sample)  # 预测均值
    y_pred = np.random.normal(mu_pred, sigma_sample, size=X.shape[0])  # 从预测分布采样
    S_new.append(gamma_sample*beta_sample)
    beta_new_look.append(S_sample)
    gamma_new.append(gamma_sample)
    predictions.append(y_pred)
S_mean_new = np.array(S_new).mean(axis=0)
beta_new_look_new = np.array(beta_new_look).mean(axis=0)
gamma_mean = np.array(gamma_new).mean(axis=0) 
gamma_binary = (gamma_mean > 0.8).astype(int)
print(gamma_mean)
# 转置为 (新样本数 x 后验样本数)
predictions = np.array(predictions).T  # (num_samples x 新样本数)
pred_mean = predictions.mean(axis=1) 
print(S_mean_new)
print(beta_new_look_new)
SS_res = np.sum((pred_mean - y) ** 2)
SS_tot = np.sum((y - np.mean(y)) ** 2)
R_squared = 1 - (SS_res / SS_tot)
print(f"模型的 R² 值为: {R_squared:.4f}")

[0.248 1.    0.131 0.698 0.212 0.469 0.654 0.995 0.261 0.08  0.525 0.162
 0.173 0.084 0.051 0.594]
[ 0.1022711   0.54676745 -0.0048244   0.15066552  0.05378714 -0.19748289
  0.72997159 -0.30826272 -0.05464017 -0.0016075   0.320185    0.00407549
 -0.01271322  0.00338075  0.00077533  0.33666224]
[7.557 6.396 7.763 6.649 7.503 8.007 7.649 6.888 7.999 7.553 7.546 7.735
 7.966 7.588 7.762 8.331]
模型的 R² 值为: 0.6176


In [13]:
import matplotlib.pyplot as plt
import numpy as np

# 假设的 S 数据
np.random.seed(42)
S_new = np.random.randint(1, 11, size=(1000, 15))  # 10 行，15 列，每列是类别 (1 到 10)

# 绘制每列的分布
num_params = S_new.shape[1]  # 参数的数量
fig, axes = plt.subplots(3, 5, figsize=(15, 10))  # 3 行 5 列的子图

for i in range(num_params):
    ax = axes[i // 5, i % 5]
    ax.hist(S_new[:, i], bins=np.arange(1, 12) - 0.5, edgecolor='black', alpha=0.75)
    ax.set_title(f"Param {i+1}")
    ax.set_xticks(range(1, 11))
    ax.set_xlabel("Category")
    ax.set_ylabel("Frequency")

plt.tight_layout()
plt.show()


<IPython.core.display.Javascript object>

In [152]:
S_new[800:900]

[array([ 7,  7,  5, 15, 13,  5, 12,  5,  7, 15,  7, 11, 11,  6, 15, 13]),
 array([10,  5, 12,  7,  2,  5,  6, 13,  6,  5,  2,  5,  6,  6,  6,  3]),
 array([ 2, 11,  2,  2, 11, 11,  2,  3, 10,  2, 11, 10,  2,  2, 10,  2]),
 array([ 1,  1,  1,  1,  1,  8, 14,  8,  8,  1,  1,  1,  8,  8,  8,  1]),
 array([10,  2,  4,  2,  3,  4, 10,  3,  3,  4,  8, 10,  4,  4,  4,  2]),
 array([15, 15,  4, 15, 15, 11,  5, 12,  6,  5, 15,  7, 11,  4,  5, 15]),
 array([ 5,  7,  0,  7, 12,  5,  1, 14, 14,  7, 14,  5,  7,  7,  5,  5]),
 array([ 0, 11, 11,  2,  2,  2,  2,  0, 11, 11, 11, 11,  0, 11, 11,  0]),
 array([12,  0, 14, 12,  7, 12,  4, 12, 12, 12,  0,  5, 14, 14,  8, 13]),
 array([15, 15,  6, 15, 15, 15,  5,  0, 15, 15, 15,  4, 11, 15, 15, 15]),
 array([15, 15,  4, 15, 15, 15,  3, 12,  4,  4, 15, 11, 11,  4, 15, 15]),
 array([ 0,  0, 15,  0,  7, 15,  7,  3, 12, 12,  2,  2,  3,  2,  3,  2]),
 array([ 2,  2, 13,  8, 13,  6, 13, 13, 14,  2,  2, 13,  6, 13, 15,  6]),
 array([ 1,  1,  1,  1,  2,  1,  1,  1

In [128]:
S_new_df = pd.DataFrame(S_new, columns=[f"Param{i+1}" for i in range(S_new.shape[1])])
S_new_df.to_excel("S_new.xlsx", index=False)

In [1]:
az.plot_trace(trace, var_names=["beta", "gamma", "sigma2", "eta", "pi", "p"], figsize=(12, 20))


NameError: name 'az' is not defined

In [14]:
print(trace)

Inference data with groups:
	> posterior
	> sample_stats
	> observed_data


In [15]:
print(trace.posterior)



<xarray.Dataset> Size: 11MB
Dimensions:       (chain: 4, draw: 5000, S_dim_0: 15, beta_m_dim_0: 10,
                   gamma_dim_0: 15, p_dim_0: 10, beta_dim_0: 15)
Coordinates:
  * chain         (chain) int64 32B 0 1 2 3
  * draw          (draw) int64 40kB 0 1 2 3 4 5 ... 4995 4996 4997 4998 4999
  * S_dim_0       (S_dim_0) int64 120B 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14
  * beta_m_dim_0  (beta_m_dim_0) int64 80B 0 1 2 3 4 5 6 7 8 9
  * gamma_dim_0   (gamma_dim_0) int64 120B 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14
  * p_dim_0       (p_dim_0) int64 80B 0 1 2 3 4 5 6 7 8 9
  * beta_dim_0    (beta_dim_0) int64 120B 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14
Data variables:
    S             (chain, draw, S_dim_0) int64 2MB 7 7 4 1 1 1 1 ... 3 6 6 3 5 0
    beta_m        (chain, draw, beta_m_dim_0) float64 2MB 0.2822 ... 0.05692
    gamma         (chain, draw, gamma_dim_0) int64 2MB 1 1 1 1 1 1 ... 1 0 1 0 1
    alpha         (chain, draw) float64 160kB 0.2416 0.3413 ... 0.8638 0.9114
    p             

In [16]:
import matplotlib.pyplot as plt

# Loop through each beta index
for i in range(5):  # beta_dim_0 has size 5
    az.plot_trace(
        trace,
        var_names=["beta"],
        coords={"beta_dim_0": i},  # Indexing beta_dim_0
    )
    plt.suptitle(f"Trace and Posterior for beta[{i}]", y=1.02)
    plt.show()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>